In [ ]:
# activity_pyspark_QUESTION is the file that contains the questions for the activity.
# retail_csv is the dataset used.


import pyspark
from pyspark.sql.functions import sum
from pyspark.sql.functions import col
from pyspark.sql.functions import desc, avg

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Retail Sales Analysis").getOrCreate()

df = spark.read.csv("./retail_sales.csv", header=True, inferSchema=True)
df.show()
df.printSchema()

+--------------+--------+----------+-----------+----------+--------+----------+
|Transaction_ID|Customer|   Product|   Category|     Store|Quantity|Unit_Price|
+--------------+--------+----------+-----------+----------+--------+----------+
|          T101|    C001|    Laptop|Electronics|     Kochi|       1|     55000|
|          T102|    C002|     Mouse|Accessories|Trivandrum|       2|       500|
|          T103|    C003|  Keyboard|Accessories|     Kochi|       1|      1500|
|          T104|    C001|Headphones|Electronics|   Calicut|       2|      2000|
|          T105|    C004|    Laptop|Electronics|Trivandrum|       1|     55000|
|          T106|    C005|     Mouse|Accessories|     Kochi|       3|       500|
|          T107|    C002|   Monitor|Electronics|   Calicut|       1|     12000|
|          T108|    C006|  Keyboard|Accessories|Trivandrum|       2|      1500|
|          T109|    C003|    Laptop|Electronics|     Kochi|       1|     55000|
|          T110|    C004|     Mouse|Acce

In [ ]:
# Display the first five records.
df.show(5)

+--------------+--------+----------+-----------+----------+--------+----------+
|Transaction_ID|Customer|   Product|   Category|     Store|Quantity|Unit_Price|
+--------------+--------+----------+-----------+----------+--------+----------+
|          T101|    C001|    Laptop|Electronics|     Kochi|       1|     55000|
|          T102|    C002|     Mouse|Accessories|Trivandrum|       2|       500|
|          T103|    C003|  Keyboard|Accessories|     Kochi|       1|      1500|
|          T104|    C001|Headphones|Electronics|   Calicut|       2|      2000|
|          T105|    C004|    Laptop|Electronics|Trivandrum|       1|     55000|
+--------------+--------+----------+-----------+----------+--------+----------+
only showing top 5 rows


In [36]:
# Find the total number of transactions in the dataset.

df.count()

15

In [31]:
# Create a new column named Total_Amount by multiplying Quantity by Unit_Price.
df = df.withColumn("Total_Amount", df["Quantity"] * df["Unit_Price"])
df.show()

+--------------+--------+----------+-----------+----------+--------+----------+------------+
|Transaction_ID|Customer|   Product|   Category|     Store|Quantity|Unit_Price|Total_Amount|
+--------------+--------+----------+-----------+----------+--------+----------+------------+
|          T101|    C001|    Laptop|Electronics|     Kochi|       1|     55000|       55000|
|          T102|    C002|     Mouse|Accessories|Trivandrum|       2|       500|        1000|
|          T103|    C003|  Keyboard|Accessories|     Kochi|       1|      1500|        1500|
|          T104|    C001|Headphones|Electronics|   Calicut|       2|      2000|        4000|
|          T105|    C004|    Laptop|Electronics|Trivandrum|       1|     55000|       55000|
|          T106|    C005|     Mouse|Accessories|     Kochi|       3|       500|        1500|
|          T107|    C002|   Monitor|Electronics|   Calicut|       1|     12000|       12000|
|          T108|    C006|  Keyboard|Accessories|Trivandrum|       2|  

In [9]:
# Display only the transactions where Total_Amount is greater than ₹10,000.
df = df.filter(df["Total_Amount"] > 10000)
df.show()

+--------------+--------+-------+-----------+----------+--------+----------+------------+------------+
|Transaction_ID|Customer|Product|   Category|     Store|Quantity|Unit_Price|Total Amount|Total_Amount|
+--------------+--------+-------+-----------+----------+--------+----------+------------+------------+
|          T101|    C001| Laptop|Electronics|     Kochi|       1|     55000|       55000|       55000|
|          T105|    C004| Laptop|Electronics|Trivandrum|       1|     55000|       55000|       55000|
|          T107|    C002|Monitor|Electronics|   Calicut|       1|     12000|       12000|       12000|
|          T109|    C003| Laptop|Electronics|     Kochi|       1|     55000|       55000|       55000|
|          T111|    C007|Monitor|Electronics|     Kochi|       2|     12000|       24000|       24000|
|          T115|    C007| Laptop|Electronics|   Calicut|       1|     55000|       55000|       55000|
+--------------+--------+-------+-----------+----------+--------+--------

In [37]:
# Identify and display any duplicate transaction records, if present.
df = df.groupBy("Transaction_ID").count().filter(col("count") > 1)
df.show()

total_count = df.count()
distinct_count = df.distinct().count()

duplicate_count = total_count - distinct_count
print(f"Total duplicate rows: {duplicate_count}")

+--------------+-----+
|Transaction_ID|count|
+--------------+-----+
+--------------+-----+

Total duplicate rows: 0


In [20]:
# 8. Calculate the total revenue generated by each product.
product_revenue = df.groupBy("Product").agg(
    sum("Total_Amount").alias("Total_Revenue")
)

product_revenue.show()

+----------+-------------+
|   Product|Total_Revenue|
+----------+-------------+
|    Laptop|       220000|
|     Mouse|         5500|
|  Keyboard|         6000|
|   Monitor|        36000|
|Headphones|         6000|
+----------+-------------+



In [22]:
# 9. Calculate the total revenue generated by each store.
store_revenue = df.groupBy("Store").agg(
    sum("Total_Amount").alias("Total_Revenue")
)

store_revenue.show()

+----------+-------------+
|     Store|Total_Revenue|
+----------+-------------+
|     Kochi|       138000|
|   Calicut|        74500|
|Trivandrum|        61000|
+----------+-------------+



In [24]:
# 10. Find the total quantity sold for each product category.
Total_Quantity_Sold = df.groupBy("Category").agg(
    sum("Quantity").alias("Total_Quantity_Sold"))
Total_Quantity_Sold.show()

+-----------+-------------------+
|   Category|Total_Quantity_Sold|
+-----------+-------------------+
|Electronics|                 10|
|Accessories|                 15|
+-----------+-------------------+



In [ ]:
# 11. Identify the product that generated the highest total revenue.



highest_revenue_product = df.groupBy("Product").agg(
    sum("Total_Amount").alias("Total_Revenue")
).orderBy(desc("Total_Revenue"))
highest_revenue_product.show(1)

+-------+-------------+
|Product|Total_Revenue|
+-------+-------------+
| Laptop|       220000|
+-------+-------------+
only showing top 1 row


In [33]:
# 12. Calculate the average transaction amount for each store.
average_transaction = df.groupBy("Store") \
    .agg(
        avg("Total_Amount").alias("Average_Transaction_Amount")
    ) \
    .orderBy(desc("Average_Transaction_Amount"))
average_transaction.show()

+----------+--------------------------+
|     Store|Average_Transaction_Amount|
+----------+--------------------------+
|     Kochi|                   23000.0|
|Trivandrum|                   15250.0|
|   Calicut|                   14900.0|
+----------+--------------------------+



In [34]:
# 13. Find the top three customers based on their total spending
top_customers = df.groupBy("Customer") \
    .agg(
        sum("Total_Amount").alias("Total_Spending")
    ) \
    .orderBy(desc("Total_Spending")) \
    .limit(3)

top_customers.show()

+--------+--------------+
|Customer|Total_Spending|
+--------+--------------+
|    C007|         79000|
|    C001|         60500|
|    C004|         57000|
+--------+--------------+

